# Spatial thin-plate spline lab

Fit a smooth surface to synthetic measurements at projected planar locations.

In [ ]:
import sys
sys.path.insert(0, "implementations/python/spatial-smoothing")
import numpy as np
from thin_plate_spline import ThinPlateSpline2D
from thin_plate_regression_spline import ThinPlateRegressionSpline2D
rng = np.random.default_rng(12)
xy = rng.uniform(-3, 3, size=(60, 2))
truth = np.sin(xy[:, 0]) + 0.4 * xy[:, 1] ** 2
observed = truth + rng.normal(0, 0.2, size=xy.shape[0])


In [ ]:
model = ThinPlateSpline2D(smoothing=0.8).fit(xy, observed)
grid = np.array([[0., 0.], [1., 1.], [-1., 2.]])
print('predictions:', model.predict(grid))
print('training RMSE:', np.sqrt(np.mean((model.predict(xy) - observed) ** 2)))


## Interpretation
Change the smoothing parameter and inspect the bias/variance trade-off. This is a full-knot thin-plate smoothing spline, not the low-rank thin-plate regression spline used in scalable GAM software. Use projected coordinates and do not expect the global kernel to respect coastlines or domain barriers.

## Low-rank thin-plate regression spline basis

The TPRS constructor transforms the full thin-plate basis and truncates its eigen-directions. k counts penalized radial terms; the affine null space adds three more coefficients.

In [ ]:
k = 12
tprs = ThinPlateRegressionSpline2D(k=k, smoothing=0.8).fit(xy, observed)
print('basis rank:', tprs.basis_rank_)
print('total coefficient dimension:', tprs.basis_dimension_)
print('predictions:', tprs.predict(grid))
print('training RMSE:', np.sqrt(np.mean((tprs.predict(xy) - observed) ** 2)))


## Full-rank equivalence check

When k = n - 3, all penalized eigen-directions are retained. The transformed reduced basis should reproduce the full-knot smoother, up to floating-point tolerance.

In [ ]:
full_rank_tprs = ThinPlateRegressionSpline2D(k=xy.shape[0] - 3, smoothing=0.8).fit(xy, observed)
full_knot = ThinPlateSpline2D(smoothing=0.8).fit(xy, observed)
print('maximum absolute difference:', np.max(np.abs(full_rank_tprs.predict(grid) - full_knot.predict(grid))))
